In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
# 画像を読み込み
x_train = np.load('Data/TrainData_NoBed.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))

print('Resized train vol_shape:', x_train.shape[1:])
print('Resized train shape:', x_train.shape)

Resized train vol_shape: (128, 256, 256)
Resized train shape: (400, 128, 256, 256)


In [5]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        # ファインチューニングでは同じ症例同士のペアを避ける
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        while np.any(idx2 == idx1):
            same_case = idx2 == idx1
            idx2[same_case] = np.random.randint(0, x_data.shape[0], size=same_case.sum())
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [6]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 256, 256])
Fixed Images Shape: torch.Size([2, 1, 128, 256, 256])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 256, 256])
Zero Gradient Shape: (2, 128, 256, 256, 3)


In [7]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    y_true = y_true.to(device)
    y_pred = y_pred.to(device)
    mse = mse_loss(y_true, y_pred)
    return mse

def lncc_loss(I, J, window=9, eps=1e-5):
    # I, J: (B, 1, D, H, W)
    padding = window // 2
    weight = torch.ones(1, 1, window, window, window, device=I.device)

    I2 = I * I
    J2 = J * J
    IJ = I * J

    I_sum = F.conv3d(I, weight, padding=padding)
    J_sum = F.conv3d(J, weight, padding=padding)
    I2_sum = F.conv3d(I2, weight, padding=padding)
    J2_sum = F.conv3d(J2, weight, padding=padding)
    IJ_sum = F.conv3d(IJ, weight, padding=padding)

    win_size = window ** 3
    u_I = I_sum / win_size
    u_J = J_sum / win_size

    cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
    I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
    J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size

    lncc = cross * cross / (I_var * J_var + eps)
    return -torch.mean(lncc)  # maximize LNCC → minimize -LNCC

In [8]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [9]:
import voxelmorph as vxm
import inspect

print(vxm.__file__)
print(vxm.networks.__file__)
print([name for name in dir(vxm.networks) if "VxmDense" in name])

d:\Saito\voxelmorph\__init__.py
d:\Saito\voxelmorph\torch\networks.py
['VxmDense', 'VxmDense1', 'VxmDense2', 'VxmDense_128_256', 'VxmDense_128_256_256']


In [10]:
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0)
model3D.to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-4)

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

[64, 128, 128]


c:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [11]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(
    w_up,
    synthesis_filters
):

    B, C, D, H, W = w_up.shape

    # ========================================================
    # 8帯域を1回のGrouped Conv3Dで処理
    #
    # input  : (B, 8, D, H, W)
    # filter : (8, 1, 2, 2, 2)
    # groups : 8
    #
    # 各Wavelet帯域に対応するfilterを独立して適用
    # ========================================================

    filtered_bands = F.conv3d(
        w_up,
        synthesis_filters,
        stride=1,
        padding=1,
        groups=C
    )

    # ========================================================
    # Originalと同じcrop
    # ========================================================

    filtered_bands = filtered_bands[
        :,
        :,
        :D,
        :H,
        :W
    ]

    # ========================================================
    # 8帯域を加算して再構成
    # ========================================================

    reconstructed = torch.sum(
        filtered_bands,
        dim=1,
        keepdim=True
    )

    return (
        reconstructed,
        filtered_bands
    )

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

In [12]:
# ============================================================
# Differentiable 3D Cubic Spatial Transformer
#
# source : [B, C, D, H, W]
# flow   : [B, 3, D, H, W]
#
# flow channel convention:
#   0 = z
#   1 = y
#   2 = x
#
# Cubic convolution interpolation
# 4 x 4 x 4 = 64 neighbors
#
# PyTorch only -> backprop possible
# ============================================================

import torch
import torch.nn as nn


class CubicSpatialTransformer3D(nn.Module):

    def __init__(
        self,
        size,
        a=-0.5
    ):
        super().__init__()

        self.size = tuple(size)

        # Catmull-Rom / Keys cubic parameter
        self.a = a


    # --------------------------------------------------------
    # Cubic convolution kernel
    # --------------------------------------------------------

    def cubic_kernel(
        self,
        x
    ):

        a = self.a

        ax = torch.abs(x)

        ax2 = ax * ax
        ax3 = ax2 * ax


        # |x| <= 1
        w1 = (
            (a + 2.0) * ax3
            -
            (a + 3.0) * ax2
            +
            1.0
        )


        # 1 < |x| < 2
        w2 = (
            a * ax3
            -
            5.0 * a * ax2
            +
            8.0 * a * ax
            -
            4.0 * a
        )


        return torch.where(
            ax <= 1.0,
            w1,
            torch.where(
                ax < 2.0,
                w2,
                torch.zeros_like(ax)
            )
        )


    # --------------------------------------------------------
    # Flattened gather
    # --------------------------------------------------------

    def gather_3d(
        self,
        src_flat,
        z,
        y,
        x,
        D,
        H,
        W
    ):

        B, C, _ = src_flat.shape


        # ----------------------------------------------------
        # border padding equivalent
        # ----------------------------------------------------

        z = torch.clamp(
            z,
            0,
            D - 1
        )

        y = torch.clamp(
            y,
            0,
            H - 1
        )

        x = torch.clamp(
            x,
            0,
            W - 1
        )


        index = (
            z * H * W
            +
            y * W
            +
            x
        )


        index = index.view(
            B,
            1,
            -1
        )


        index = index.expand(
            -1,
            C,
            -1
        )


        gathered = torch.gather(
            src_flat,
            2,
            index
        )


        return gathered


    # --------------------------------------------------------
    # forward
    # --------------------------------------------------------

    def forward(
        self,
        src,
        flow
    ):

        B, C, D, H, W = src.shape


        if (
            D,
            H,
            W
        ) != self.size:

            raise ValueError(
                f"Source size {src.shape[-3:]} "
                f"!= transformer size {self.size}"
            )


        device = src.device
        dtype = src.dtype


        # ====================================================
        # Base coordinate grid
        # ====================================================

        z = torch.arange(
            D,
            device=device,
            dtype=dtype
        )

        y = torch.arange(
            H,
            device=device,
            dtype=dtype
        )

        x = torch.arange(
            W,
            device=device,
            dtype=dtype
        )


        zz, yy, xx = torch.meshgrid(
            z,
            y,
            x,
            indexing="ij"
        )


        zz = zz.unsqueeze(0)
        yy = yy.unsqueeze(0)
        xx = xx.unsqueeze(0)


        # ====================================================
        # Continuous sampling locations
        # ====================================================

        sample_z = (
            zz
            +
            flow[:, 0]
        )

        sample_y = (
            yy
            +
            flow[:, 1]
        )

        sample_x = (
            xx
            +
            flow[:, 2]
        )


        # ====================================================
        # Integer base position
        # ====================================================

        z0 = torch.floor(
            sample_z
        )

        y0 = torch.floor(
            sample_y
        )

        x0 = torch.floor(
            sample_x
        )


        # ====================================================
        # Flatten source
        # ====================================================

        src_flat = src.reshape(
            B,
            C,
            -1
        )


        # ====================================================
        # output
        # ====================================================

        output = torch.zeros(
            (
                B,
                C,
                D * H * W
            ),
            device=device,
            dtype=dtype
        )


        # ====================================================
        # 4 x 4 x 4 cubic neighborhood
        #
        # offsets:
        # -1, 0, 1, 2
        # ====================================================

        for oz in (
            -1,
            0,
            1,
            2
        ):

            zi_float = (
                z0
                +
                oz
            )

            wz = self.cubic_kernel(
                sample_z
                -
                zi_float
            )


            zi = zi_float.long()


            for oy in (
                -1,
                0,
                1,
                2
            ):

                yi_float = (
                    y0
                    +
                    oy
                )

                wy = self.cubic_kernel(
                    sample_y
                    -
                    yi_float
                )


                yi = yi_float.long()


                for ox in (
                    -1,
                    0,
                    1,
                    2
                ):

                    xi_float = (
                        x0
                        +
                        ox
                    )

                    wx = self.cubic_kernel(
                        sample_x
                        -
                        xi_float
                    )


                    xi = xi_float.long()


                    # ----------------------------------------
                    # separable cubic weight
                    # ----------------------------------------

                    weight = (
                        wz
                        *
                        wy
                        *
                        wx
                    )


                    weight = weight.reshape(
                        B,
                        1,
                        -1
                    )


                    # ----------------------------------------
                    # samples
                    # ----------------------------------------

                    values = self.gather_3d(
                        src_flat,
                        zi,
                        yi,
                        xi,
                        D,
                        H,
                        W
                    )


                    output = (
                        output
                        +
                        values
                        *
                        weight
                    )


        return output.reshape(
            B,
            C,
            D,
            H,
            W
        )


print(
    "Differentiable CubicSpatialTransformer3D defined."
)

Differentiable CubicSpatialTransformer3D defined.


In [13]:
# ============================================================
# Gradient check
# ============================================================

cubic_transformer = CubicSpatialTransformer3D(
    (64, 128, 128)
).to(device)


# 小さいtensorで確認
test_src = torch.randn(
    1,
    1,
    8,
    16,
    16,
    device=device,
    requires_grad=True
)

test_flow = (
    torch.randn(
        1,
        3,
        8,
        16,
        16,
        device=device
    )
    *
    0.1
)

test_flow.requires_grad_()


test_transformer = CubicSpatialTransformer3D(
    (8, 16, 16)
).to(device)


test_out = test_transformer(
    test_src,
    test_flow
)


test_loss = (
    test_out ** 2
).mean()


test_loss.backward()


print(
    "src grad :",
    test_src.grad.abs().mean().item()
)

print(
    "flow grad:",
    test_flow.grad.abs().mean().item()
)


assert (
    test_src.grad is not None
)

assert (
    test_flow.grad is not None
)

assert (
    torch.isfinite(
        test_flow.grad
    ).all()
)


print(
    "Gradient check: OK"
)

src grad : 0.0006985047366470098
flow grad: 0.0005332543514668941
Gradient check: OK


In [14]:
# ============================================================
# DVF Smoothness loss
#
# mean squared spatial gradient
# ============================================================

def smoothness_loss_3d(flow):

    dz = (
        flow[:, :, 1:, :, :]
        -
        flow[:, :, :-1, :, :]
    )

    dy = (
        flow[:, :, :, 1:, :]
        -
        flow[:, :, :, :-1, :]
    )

    dx = (
        flow[:, :, :, :, 1:]
        -
        flow[:, :, :, :, :-1]
    )


    return (
        dz.pow(2).mean()
        +
        dy.pow(2).mean()
        +
        dx.pow(2).mean()
    ) / 3.0

In [15]:
# ============================================================
# Warp all 8 Wavelet bands
# ============================================================

def warp_wavelet_bands(
    moving_bands,
    flow,
    transformer
):

    warped_bands = torch.cat(

        [

            transformer(

                moving_bands[
                    :,
                    band:
                    band + 1
                ],

                flow
            )

            for band
            in range(
                moving_bands.shape[1]
            )
        ],

        dim=1
    )


    return warped_bands

In [16]:
# ============================================================
# Transformers
# ============================================================

transformer_trilinear_train = (
    vxm.layers.SpatialTransformer(
        (64, 128, 128),
        mode="bilinear"
    )
    .to(device)
)


transformer_cubic_train = (
    CubicSpatialTransformer3D(
        (64, 128, 128),
        a=-0.5
    )
    .to(device)
)


print(
    "Trilinear transformer ready"
)

print(
    "Cubic transformer ready"
)

Trilinear transformer ready
Cubic transformer ready


In [ ]:
# ============================================================
# SAME ARCHITECTURE / SAME PRETRAINED WEIGHTS
#
# model3D:
#   Notebook上ですでに正しい構造で定義されているモデル
#
# Trilinear / Cubic の2条件を
# 完全に同一初期重みから開始する
# ============================================================

import copy
import torch
from pathlib import Path


PRETRAINED_PATH = Path(
    r"D:\Saito\model_DVF_pretrain_final.pth"
)


if not PRETRAINED_PATH.exists():

    raise FileNotFoundError(
        f"Pretrained model not found:\n{PRETRAINED_PATH}"
    )


# ============================================================
# LOAD STATE DICT
# ============================================================

try:

    checkpoint = torch.load(
        PRETRAINED_PATH,
        map_location=device,
        weights_only=True
    )

except TypeError:

    checkpoint = torch.load(
        PRETRAINED_PATH,
        map_location=device
    )


if (
    isinstance(checkpoint, dict)
    and
    "model_state_dict" in checkpoint
):

    pretrained_state = checkpoint[
        "model_state_dict"
    ]

else:

    pretrained_state = checkpoint


# ============================================================
# COPY THE EXISTING CORRECT MODEL STRUCTURE
# ============================================================

model_trilinear = copy.deepcopy(
    model3D
).to(device)


model_cubic = copy.deepcopy(
    model3D
).to(device)


# ============================================================
# LOAD EXACTLY SAME PRETRAINED WEIGHTS
# ============================================================

model_trilinear.load_state_dict(
    pretrained_state
)


model_cubic.load_state_dict(
    pretrained_state
)


# ============================================================
# CHECK
# ============================================================

for p_tri, p_cub in zip(
    model_trilinear.parameters(),
    model_cubic.parameters()
):

    if not torch.equal(
        p_tri,
        p_cub
    ):

        raise RuntimeError(
            "Initial weights are not identical."
        )


print(
    "Same model architecture: OK"
)

print(
    "Same pretrained weights: OK"
)

print(
    "Trilinear model ready"
)

print(
    "Cubic model ready"
)

Steps: 5000
Smooth lambda: 0.02


In [18]:
# ============================================================
# Create fresh model from same pretrained weight
# ============================================================

def create_pretrained_model():

    model = vxm.networks.VxmDense_128_256_256(

        (128, 256, 256),

        nb_features=nb_features,

        int_steps=0
    ).to(device)


    checkpoint = torch.load(
        PRETRAINED_PATH,
        map_location=device
    )


    if (
        isinstance(
            checkpoint,
            dict
        )
        and
        "model_state_dict"
        in checkpoint
    ):

        state_dict = checkpoint[
            "model_state_dict"
        ]

    else:

        state_dict = checkpoint


    model.load_state_dict(
        state_dict
    )


    return model


model_trilinear = create_pretrained_model()

model_cubic = create_pretrained_model()


print(
    "Same pretrained weights loaded."
)


# 完全一致確認
for (
    p_tri,
    p_cub
) in zip(
    model_trilinear.parameters(),
    model_cubic.parameters()
):

    assert torch.equal(
        p_tri,
        p_cub
    )


print(
    "Initial weights identical: OK"
)

TypeError: VxmDense_128_256_256.__init__() got an unexpected keyword argument 'nb_features'

In [ ]:
# ============================================================
# FIXED 360 / 40 SPLIT
# ============================================================

rng_split = np.random.RandomState(
    SEED
)


all_indices = np.arange(
    400
)


rng_split.shuffle(
    all_indices
)


train_indices = all_indices[
    :360
]


val_indices = all_indices[
    360:
]


print(
    "Train:",
    len(train_indices)
)

print(
    "Val  :",
    len(val_indices)
)

In [ ]:
# ============================================================
# SAME PAIR SEQUENCE
#
# Both conditions get exactly same Moving / Fixed pairs
# ============================================================

rng_pair = np.random.RandomState(
    12345
)


training_pair_sequence = []


for step in range(
    NUM_STEPS
):

    moving_ids = rng_pair.choice(
        train_indices,
        size=BATCH_SIZE,
        replace=True
    )


    fixed_ids = rng_pair.choice(
        train_indices,
        size=BATCH_SIZE,
        replace=True
    )


    # 同一患者を避ける
    for b in range(
        BATCH_SIZE
    ):

        while (
            fixed_ids[b]
            ==
            moving_ids[b]
        ):

            fixed_ids[b] = rng_pair.choice(
                train_indices
            )


    training_pair_sequence.append(
        (
            moving_ids.copy(),
            fixed_ids.copy()
        )
    )


print(
    "Pair sequence:",
    len(
        training_pair_sequence
    )
)

In [ ]:
# ============================================================
# Get batch
#
# x_train:
# [400, 128, 256, 256]
# ============================================================

def get_training_batch(
    moving_ids,
    fixed_ids
):

    moving = torch.from_numpy(
        np.asarray(
            x_train[
                moving_ids
            ],
            dtype=np.float32
        )
    )


    fixed = torch.from_numpy(
        np.asarray(
            x_train[
                fixed_ids
            ],
            dtype=np.float32
        )
    )


    # [B,D,H,W]
    # ->
    # [B,1,D,H,W]

    moving = moving.unsqueeze(
        1
    ).to(device)


    fixed = fixed.unsqueeze(
        1
    ).to(device)


    return (
        moving,
        fixed
    )

In [ ]:
# ============================================================
# Train one condition
# ============================================================

def train_one_interpolation_condition(
    name,
    model,
    transformer,
    pair_sequence
):

    print()
    print(
        "=" * 100
    )

    print(
        f"TRAIN: {name}"
    )

    print(
        "=" * 100
    )


    model.train()


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )


    history = []


    start_time = time.time()


    for step in range(
        1,
        NUM_STEPS + 1
    ):


        moving_ids, fixed_ids = (
            pair_sequence[
                step - 1
            ]
        )


        # ====================================================
        # Wavelet data
        # ====================================================

        moving_bands, fixed_bands = (
            get_wavelet_batch(
                moving_ids,
                fixed_ids
            )
        )


        # ====================================================
        # Forward
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        flow = model(
            moving_bands,
            fixed_bands
        )


        warped_bands = warp_wavelet_bands(
            moving_bands,
            flow,
            transformer_trilinear_train
        )


        # ====================================================
        # Similarity in Wavelet space
        #
        # 同じ条件比較なので、
        # 学習時に今まで使っていたMSE定義に
        # 合わせるのが最優先
        # ====================================================

        mse_loss = F.mse_loss(
            warped_bands.float(),
            fixed_bands.float()
        )


        smooth_loss = smoothness_loss_3d(
            flow.float()
        )


        total_loss = (
            mse_loss
            +
            SMOOTH_LAMBDA
            *
            smooth_loss
        )


        # ====================================================
        # Backward
        # ====================================================

        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        # ====================================================
        # History
        # ====================================================

        if (
            step == 1
            or
            step % 100 == 0
        ):

            elapsed = (
                time.time()
                -
                start_time
            )


            row = {

                "step":
                    step,

                "loss":
                    float(
                        total_loss.item()
                    ),

                "mse":
                    float(
                        mse_loss.item()
                    ),

                "smooth":
                    float(
                        smooth_loss.item()
                    ),

                "elapsed_min":
                    elapsed / 60.0
            }


            history.append(
                row
            )


            print(
                f"{name} | "
                f"{step:05d}/{NUM_STEPS} | "
                f"Loss={row['loss']:.8f} | "
                f"MSE={row['mse']:.8f} | "
                f"Smooth={row['smooth']:.8f} | "
                f"{row['elapsed_min']:.1f} min"
            )


    # ========================================================
    # SAVE MODEL
    # ========================================================

    model_path = (
        OUTPUT_DIR_CUBIC_TRAIN
        /
        f"{name}_5000.pth"
    )


    torch.save(
        model.state_dict(),
        model_path
    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history_df = pd.DataFrame(
        history
    )


    history_path = (
        OUTPUT_DIR_CUBIC_TRAIN
        /
        f"{name}_history.csv"
    )


    history_df.to_csv(
        history_path,
        index=False
    )


    print()
    print(
        "Saved model:",
        model_path
    )

    print(
        "Saved history:",
        history_path
    )


    return (
        model,
        history_df
    )

In [ ]:
model_trilinear, history_tri = (
    train_one_interpolation_condition(

        name="Trilinear",

        model=model_trilinear,

        transformer=transformer_trilinear_train,

        pair_sequence=training_pair_sequence
    )
)

In [ ]:
model_cubic, history_cubic = (
    train_one_interpolation_condition(

        name="Cubic",

        model=model_cubic,

        transformer=transformer_cubic_train,

        pair_sequence=training_pair_sequence
    )
)

In [ ]:
# ============================================================
# Learning curves
# ============================================================

import matplotlib.pyplot as plt


plt.figure(
    figsize=(9, 5)
)


plt.plot(
    history_tri[
        "step"
    ],
    history_tri[
        "mse"
    ],
    label="Trilinear"
)


plt.plot(
    history_cubic[
        "step"
    ],
    history_cubic[
        "mse"
    ],
    label="Cubic"
)


plt.xlabel(
    "Training step"
)

plt.ylabel(
    "MSE loss"
)

plt.title(
    "5000-step training\n"
    "Trilinear vs Cubic"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()